# Model 1: train the grammar detector

The detector answers three questions: is there a target error, what category is it, and where is it?
RoBERTa supplies fixed token representations. A bidirectional long short-term memory network (Bi-LSTM)
reads the sentence in both directions. Attention combines the token states for sentence-level predictions.
Only the Bi-LSTM, attention, and output layers learn during this training.

Run the cells in order. Supply prepared private train and validation files. No private examples or outputs
are included here. The architecture and alignment code is visible below. The website uses matching code
in `models/`. Tests check that these definitions agree. A new run can produce different weights and scores.

## 1. Set paths and training settings
The defaults match the research training command. Use the settings from the run you want to reproduce.

In [ ]:
from pathlib import Path
import json
import random
import torch

# Start Jupyter from the project root.
assert Path("models/model1.py").is_file(), "Open Jupyter from the PatternLab folder."
PRIVATE_ROOT = Path.home() / "patternlab-private"

from dataclasses import dataclass, asdict
from functools import partial
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import AutoTokenizer
from models.model1 import CATEGORIES, ENCODER_REVISION, encoder_revision
from models.data import read_jsonl, validate_record, SCHEMA_VERSION
from models.checkpoints import model_inputs

TRAIN_FILE = PRIVATE_ROOT / "model1" / "train.jsonl"
VALIDATION_FILE = PRIVATE_ROOT / "model1" / "validation.jsonl"
OUTPUT = PRIVATE_ROOT / "runs" / "model1-presentation"
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 256

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Load the prepared records
Each positive record has one category and a character span. A missing word has equal start and end positions.
Negative records have no category or span. The loader checks the file format and split labels.
Train and validation sentences must not overlap. We do not use test data to select a checkpoint.

In [ ]:
assert TRAIN_FILE.is_file() and VALIDATION_FILE.is_file(), "Set the two private data paths first."
assert not OUTPUT.exists(), "Use a new output folder to protect previous runs."
train_records = read_jsonl(TRAIN_FILE, "train")
validation_records = read_jsonl(VALIDATION_FILE, "validation")
train_texts = {record["text"].strip().casefold() for record in train_records}
assert not any(record["text"].strip().casefold() in train_texts for record in validation_records)
print({"training_records": len(train_records), "validation_records": len(validation_records)})

## 3. Convert character labels to token boundaries
A tokenizer divides text into smaller units called tokens. Its offset map connects each token to the
original text. We preserve exact alignment. We reject long or misaligned examples instead of cutting them.
The dataset also removes special tokens from the locations that the model can predict.

In [ ]:
class SentenceDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=256):
        self.items = []
        if not tokenizer.is_fast:
            raise ValueError("A fast tokenizer with offset mappings is required")
        seen = set()
        for number, record in enumerate(records, 1):
            try:
                validate_record(record)
                key = record["text"].strip().casefold()
                if key in seen:
                    raise ValueError("Duplicate sentence; v1 allows one focal annotation per sentence")
                seen.add(key)
                encoded = tokenizer(record["text"], return_offsets_mapping=True,
                                    return_special_tokens_mask=True, truncation=False)
                if len(encoded["input_ids"]) > max_length:
                    raise ValueError("Example exceeds max_length; split or exclude it explicitly")
                indices = [i for i, (offset, special) in enumerate(zip(
                    encoded["offset_mapping"], encoded["special_tokens_mask"]))
                    if not special and offset[1] > offset[0]]
                offsets = [tuple(encoded["offset_mapping"][i]) for i in indices]
                if not offsets:
                    raise ValueError("No content tokens")
                start = end = category = -100
                if record["has_error"]:
                    a, b = record["span"]
                    starts = [x[0] for x in offsets] + [offsets[-1][1]]
                    if a == b:
                        if a not in starts:
                            raise ValueError("Insertion must align to a token start or final token end")
                        start = end = starts.index(a)
                    else:
                        ends = [x[1] for x in offsets]
                        if a not in starts[:-1] or b not in ends:
                            raise ValueError("Span must align exactly to token edges")
                        start, end = starts.index(a), len(ends) - ends[::-1].index(b)
                        if end <= start:
                            raise ValueError("Invalid token span")
                    category = CATEGORIES.index(record["category"])
                self.items.append({"input_ids": encoded["input_ids"],
                    "content_indices": indices, "offsets": offsets,
                    "has_error": int(record["has_error"]), "category": category,
                    "start": start, "end": end})
            except (ValueError, KeyError, TypeError) as error:
                # Never put private learner text in exception messages.
                raise ValueError(f"Example {number}: {error}") from error
        if not self.items:
            raise ValueError("Dataset is empty")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

## 4. Put examples into batches
Sentences have different lengths. Padding fills the unused positions in each batch. Masks tell the model
which positions contain real tokens. Padding must not affect attention or the error location.

In [ ]:
def collate(items, pad_token_id):
    batch_size = len(items)
    width = max(len(item["input_ids"]) for item in items)
    tokens = max(len(item["content_indices"]) for item in items)
    batch = {"input_ids": torch.full((batch_size, width), pad_token_id, dtype=torch.long),
             "attention_mask": torch.zeros(batch_size, width, dtype=torch.long),
             "content_indices": torch.zeros(batch_size, tokens, dtype=torch.long),
             "content_mask": torch.zeros(batch_size, tokens, dtype=torch.bool)}
    for row, item in enumerate(items):
        length, count = len(item["input_ids"]), len(item["content_indices"])
        batch["input_ids"][row, :length] = torch.tensor(item["input_ids"])
        batch["attention_mask"][row, :length] = 1
        batch["content_indices"][row, :count] = torch.tensor(item["content_indices"])
        batch["content_mask"][row, :count] = True
    for key in ("has_error", "category", "start", "end"):
        batch[key] = torch.tensor([item[key] for item in items], dtype=torch.long)
    return batch

## 5. Define the network
The frozen encoder does not receive gradients. The Bi-LSTM reads only the content tokens.
Attention weights sum to one over real tokens. Detection and category use the pooled sentence state.
Location uses start and end boundary scores. Equal boundaries represent a missing word.
The layer names stay compatible with the existing checkpoints.

In [ ]:
@dataclass
class ModelConfig:
    encoder_name: str = "FacebookAI/roberta-base"
    encoder_revision: str | None = None
    hidden_size: int = 128
    dropout: float = 0.2
    max_length: int = 256


class Detector(nn.Module):
    def __init__(self, encoder, config=None):
        super().__init__()
        self.config = config or ModelConfig()
        self.encoder = encoder
        self.encoder.requires_grad_(False)
        self.encoder.eval()
        self.lstm = nn.LSTM(encoder.config.hidden_size, self.config.hidden_size,
                            batch_first=True, bidirectional=True)
        width = 2 * self.config.hidden_size
        self.dropout = nn.Dropout(self.config.dropout)
        self.attention = nn.Linear(width, 1)
        self.detection = nn.Linear(width, 1)
        self.category = nn.Linear(width, len(CATEGORIES))
        # A boundary joins the states on either side, including sentence edges.
        self.start = nn.Linear(2 * width, 1)
        self.end = nn.Linear(2 * width, 1)

    @classmethod
    def from_pretrained(cls, config=None, *, local_files_only=False):
        from transformers import AutoModel
        config = config or ModelConfig()
        return cls(AutoModel.from_pretrained(config.encoder_name,
                   revision=encoder_revision(config.encoder_name, config.encoder_revision), local_files_only=local_files_only), config)

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()  # Freeze dropout as well as weights.
        return self

    def forward(self, input_ids, attention_mask, content_indices, content_mask):
        with torch.no_grad():
            encoded = self.encoder(input_ids=input_ids,
                                   attention_mask=attention_mask).last_hidden_state
        states = encoded.gather(1, content_indices.unsqueeze(-1).expand(
            -1, -1, encoded.shape[-1]))
        lengths = content_mask.sum(1)
        if (lengths == 0).any():
            raise ValueError("Sentences must contain at least one content token")
        packed = pack_padded_sequence(states, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        sequence, _ = self.lstm(packed)
        sequence, _ = pad_packed_sequence(sequence, batch_first=True,
                                         total_length=content_mask.shape[1])
        sequence = self.dropout(sequence) * content_mask.unsqueeze(-1)
        scores = self.attention(sequence).squeeze(-1).masked_fill(~content_mask, -torch.inf)
        attention = scores.softmax(-1)
        pooled = (sequence * attention.unsqueeze(-1)).sum(1)
        edge = sequence.new_zeros(sequence.shape[0], 1, sequence.shape[-1])
        boundaries = torch.cat((torch.cat((edge, sequence), 1),
                                torch.cat((sequence, edge), 1)), -1)
        boundary_mask = torch.arange(boundaries.shape[1], device=lengths.device)[None] <= lengths[:, None]
        return {
            "detection_logits": self.detection(pooled).squeeze(-1),
            "category_logits": self.category(pooled),
            "start_logits": self.start(boundaries).squeeze(-1).masked_fill(~boundary_mask, -torch.inf),
            "end_logits": self.end(boundaries).squeeze(-1).masked_fill(~boundary_mask, -torch.inf),
            "boundary_mask": boundary_mask,
            "attention": attention,
        }

## 6. Define the loss and decode the predictions
Loss measures prediction error. Detection uses binary cross-entropy. Category and location use
cross-entropy only for positive examples. Total loss = detection + category + location.
Decoding chooses one valid start/end pair. The detection threshold here is 0.5. A deployment threshold
must come from the chosen checkpoint's validation results.

In [ ]:
def multitask_loss(outputs, batch):
    """Category/localization supervision only exists on positive examples."""
    detection = nn.functional.binary_cross_entropy_with_logits(
        outputs["detection_logits"], batch["has_error"].float())
    positive = batch["has_error"].bool()
    # Avoid summing masked -inf logits when an entire batch is correct.
    zero = outputs["category_logits"].sum() * 0
    category = localization = zero
    if positive.any():
        category = nn.functional.cross_entropy(outputs["category_logits"][positive],
                                                batch["category"][positive])
        localization = (nn.functional.cross_entropy(outputs["start_logits"][positive], batch["start"][positive])
                        + nn.functional.cross_entropy(outputs["end_logits"][positive], batch["end"][positive])) / 2
    return {"total": detection + category + localization,
            "detection": detection, "category": category, "localization": localization}


def decode(outputs, threshold=0.5):
    """Select one valid start <= end pair; equality denotes an insertion."""
    if not 0 < threshold < 1:
        raise ValueError("threshold must be between zero and one")
    probability = outputs["detection_logits"].sigmoid()
    joint = outputs["start_logits"].unsqueeze(2) + outputs["end_logits"].unsqueeze(1)
    n = joint.shape[1]
    valid = torch.ones(n, n, dtype=torch.bool, device=joint.device).triu()
    joint = joint.masked_fill(~valid, -torch.inf)
    index = joint.flatten(1).argmax(1)
    return {"has_error": probability >= threshold, "probability": probability,
            "category": outputs["category_logits"].argmax(1),
            "start": index // n, "end": index % n}

## 7. Load RoBERTa and prepare the optimizer
The model revision fixes which pretrained weights we load. AdamW updates only the trainable parameters.
The seeded data loader shuffles the training set. The validation loader keeps its order.

In [ ]:
config = ModelConfig(encoder_revision=ENCODER_REVISION, max_length=MAX_LENGTH)
tokenizer = AutoTokenizer.from_pretrained(config.encoder_name, revision=ENCODER_REVISION, use_fast=True)
model = Detector.from_pretrained(config).to(DEVICE)
train_data = SentenceDataset(train_records, tokenizer, MAX_LENGTH)
validation_data = SentenceDataset(validation_records, tokenizer, MAX_LENGTH)
collator = partial(collate, pad_token_id=tokenizer.pad_token_id)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                         generator=torch.Generator().manual_seed(SEED), collate_fn=collator)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, collate_fn=collator)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
print({"device": DEVICE, "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad)})

## 8. Run one epoch
An epoch is one pass through a dataset. With an optimizer, this function computes gradients and updates
the weights. Without one, it measures validation performance. Gradient clipping limits large updates.
Category and location accuracy below are measured on known error sentences. They are not full-pipeline scores.

In [ ]:
def run_epoch(model, loader, device, optimizer=None):
    model.train(optimizer is not None)
    total_loss = count = tp = fp = fn = correct_detection = positives = correct_category = correct_location = 0
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.set_grad_enabled(optimizer is not None):
            outputs = model(**model_inputs(batch))
            losses = multitask_loss(outputs, batch)
            if not torch.isfinite(losses["total"]):
                raise RuntimeError("Non-finite loss")
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                losses["total"].backward()
                torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
                optimizer.step()
        prediction = decode(outputs)
        gold = batch["has_error"].bool()
        predicted = prediction["has_error"]
        size = len(gold)
        count += size
        total_loss += losses["total"].item() * size
        tp += (predicted & gold).sum().item()
        fp += (predicted & ~gold).sum().item()
        fn += (~predicted & gold).sum().item()
        correct_detection += (predicted == gold).sum().item()
        positives += gold.sum().item()
        # Conditional head accuracy: evaluated on gold positives, not gated by detection.
        correct_category += ((prediction["category"] == batch["category"]) & gold).sum().item()
        correct_location += ((prediction["start"] == batch["start"]) &
                             (prediction["end"] == batch["end"]) & gold).sum().item()
    return {"loss": total_loss / count, "detection_accuracy": correct_detection / count,
            "detection_f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "category_accuracy_on_gold_errors": correct_category / positives if positives else None,
            "localization_exact_on_gold_errors": correct_location / positives if positives else None,
            "examples": count}

## 9. Save compatible checkpoints
The checkpoint stores the trained layers and their settings. RoBERTa is loaded separately at the fixed revision.
The tokenizer is saved beside the checkpoint. Existing private research checkpoints use this same format.

In [ ]:
def save_checkpoint(path, model, optimizer, epoch, seed, *, synthetic=False):
    torch.save({"schema_version": SCHEMA_VERSION, "config": asdict(model.config),
                "categories": list(CATEGORIES), "head_state": {
                    k: v for k, v in model.state_dict().items() if not k.startswith("encoder.")},
                "optimizer_state": optimizer.state_dict(), "epoch": epoch, "seed": seed,
                "synthetic": synthetic}, path)

## 10. Train, validate, and save the best model
The lowest validation loss selects `best.pt`. We also save the last checkpoint and a metric history.
The output folder must be new. Training data, weights, and executed notebook outputs stay outside the public repository.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=False)
tokenizer.save_pretrained(OUTPUT / "tokenizer")
history = []
best_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, DEVICE, optimizer)
    validation_metrics = run_epoch(model, validation_loader, DEVICE)
    row = {"epoch": epoch, "train": train_metrics, "validation": validation_metrics,
           "synthetic_software_check": False}
    history.append(row)
    print(json.dumps(row))
    save_checkpoint(OUTPUT / "last.pt", model, optimizer, epoch, SEED)
    if validation_metrics["loss"] < best_loss:
        best_loss = validation_metrics["loss"]
        save_checkpoint(OUTPUT / "best.pt", model, optimizer, epoch, SEED)
(OUTPUT / "history.json").write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")

## Questions to explain in the oral defense
Why freeze RoBERTa? Which layers learn? Why use masks? How can the model locate a missing word?
Why do correct sentences have no category loss? Why select the checkpoint with validation data?
Use the running notebook to load this checkpoint. Score held-out data separately before reporting new results.